# H&M Transaction Data: Product Recommendations 03
## Decision Forest Models

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn import ensemble
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

import os
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# data processing classes
from src.customer_features import CustomerFeatureEngineer
from src.product_features import ProductFeatureEngineer
from src.recommendation_training import RecommendationTrainingBuilder


## Load Processed Data

In [12]:
# Load data from parquet
data_path = Path("../data")
processed_data_path = data_path / 'processed' / 'product_recommendation'

with open(processed_data_path / 'X_train_base.pkl', 'rb') as f:
    X_train = pickle.load(f)
with open(processed_data_path / 'y_train_base.pkl', 'rb') as f:
    y_train = pickle.load(f)
with open(processed_data_path / 'X_val_base.pkl', 'rb') as f:
    X_val = pickle.load(f)
with open(processed_data_path / 'y_val_base.pkl', 'rb') as f:
    y_val = pickle.load(f)
with open(processed_data_path / 'X_test_base.pkl', 'rb') as f:
    X_test = pickle.load(f)
with open(processed_data_path / 'y_test_base.pkl', 'rb') as f:
    y_test = pickle.load(f)

print(f"{X_train.shape=}")
print(f"{y_train.shape=}")
print(f"{X_val.shape=}")
print(f"{y_val.shape=}")
print(f"{X_test.shape=}")
print(f"{y_test.shape=}")
print()
# Check what data was loaded
print(f"X_train type: {type(X_train)}")
print(f"X_train shape: {X_train.shape}")
print(f"X_train memory: {X_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nData types:")
print(X_train.dtypes)

X_train.shape=(238936, 43)
y_train.shape=(238936,)
X_val.shape=(440082, 43)
y_val.shape=(440082,)
X_test.shape=(326736, 43)
y_test.shape=(326736,)

X_train type: <class 'pandas.core.frame.DataFrame'>
X_train shape: (238936, 43)
X_train memory: 78.39 MB

Data types:
sales_last_7_days                        float64
garment_Under-, Nightwear                  int64
avg_price                                float64
garment_Shirts                             int64
garment_Outdoor                            int64
product_price_std                        float64
garment_Jersey Basic                       int64
customer_price_std                       float64
club_member_status_NOT_ACTIVE_MEMBER     float64
category_diversity                       float64
garment_Trousers Denim                     int64
min_price                                float64
garment_Dressed                            int64
garment_Trousers                           int64
fashion_news_frequency_REGULARLY         float64

### Impute missing values

In [3]:
print("Columns with indicator value")
indicator_cols = []
for col in X_train.columns:
    col_max = X_train[col].max()
    if col_max == 999:
        indicator_cols.append(col)
        print(f"- {col}")

# Remove the indicator values and replace with NaN
# Add an indicator col instead
print("Add indicator column and prepare to impute the fill value")
for col in indicator_cols:
    for df in [X_train, X_val, X_test]:
        df[f'{col}_missing'] = (df[col] == 999).astype(int)
        df[col] = df[col].replace(999, np.nan)
        
print("Impute missing values with median strategy")
imputer = SimpleImputer(strategy='median')
X_train[indicator_cols] = imputer.fit_transform(X_train[indicator_cols])
X_val[indicator_cols] = imputer.transform(X_val[indicator_cols])
X_test[indicator_cols] = imputer.transform(X_test[indicator_cols])

Columns with indicator value
- days_since_last_purchase
- days_since_first_sale
- avg_days_between_purchases
- days_since_last_sale
Add indicator column and prepare to impute the fill value
Impute missing values with median strategy


## Model Building - Random Forest

In [8]:
random_state=67

def predict_and_report_val(model, name):
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]

    print(f"======= {name} ========")
    print(f"AUC-ROC:  {roc_auc_score(y_val, y_proba):.4f}")
    print(f"PR-AUC:   {average_precision_score(y_val, y_proba):.4f}")
    print(classification_report(y_val, y_pred, target_names=['No Purchase', 'Purchase']))

In [5]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    'n_estimators': [550, 600, 700, 800],
    'max_depth': [28, 30, 32, 36],
    'min_samples_leaf': [8, 10, 15, 20],
}

rf_search = RandomizedSearchCV(
    ensemble.RandomForestClassifier(
        random_state=random_state,
        n_jobs=-1
    ),
    param_distributions=param_distributions,
    n_iter=10,
    scoring='average_precision',
    cv=3,
    random_state=random_state,
    verbose=1
)

rf_search.fit(X_train, y_train)

print(f"Best params: {rf_search.best_params_}")
print(f"Best CV PR-AUC: {rf_search.best_score_:.4f}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'n_estimators': 550, 'min_samples_leaf': 8, 'max_depth': 30}
Best CV PR-AUC: 0.9797


In [9]:
predict_and_report_val(rf_search.best_estimator_, "Random Forest")

======= Random Forest ========
AUC-ROC:  0.9732
PR-AUC:   0.9206
              precision    recall  f1-score   support

 No Purchase       0.97      0.93      0.95    366735
    Purchase       0.73      0.87      0.79     73347

    accuracy                           0.92    440082
   macro avg       0.85      0.90      0.87    440082
weighted avg       0.93      0.92      0.93    440082



## Model Building - Gradient Boosting

In [10]:
param_distributions = {
    'max_iter': [150, 200, 250, 300],
    'max_depth': [20, 25, 30],
    'min_samples_leaf': [10, 20, 30],
    'learning_rate': [0.05, 0.1, 0.2]
}

hgb_search = RandomizedSearchCV(
    ensemble.HistGradientBoostingClassifier(random_state=67),
    param_distributions=param_distributions,
    scoring='average_precision',
    cv=3,
    random_state=random_state,
    verbose=1
)

hgb_search.fit(X_train, y_train)

print(f"Best params: {hgb_search.best_params_}")
print(f"Best CV PR-AUC: {hgb_search.best_score_:.4f}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'min_samples_leaf': 30, 'max_iter': 150, 'max_depth': 25, 'learning_rate': 0.05}
Best CV PR-AUC: 0.9808


In [11]:
predict_and_report_val(hgb_search.best_estimator_, "Gradient Boosting")

======= Gradient Boosting ========
AUC-ROC:  0.9759
PR-AUC:   0.9232
              precision    recall  f1-score   support

 No Purchase       0.98      0.92      0.95    366735
    Purchase       0.70      0.88      0.78     73347

    accuracy                           0.92    440082
   macro avg       0.84      0.90      0.87    440082
weighted avg       0.93      0.92      0.92    440082

